# KLIFS Dataset Visualization

Exploración interactiva del dataset de kinasas descargadas desde KLIFS.
Este notebook permite visualizar y analizar las estructuras y sus propiedades.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Configuración de estilo
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

# Cargar dataset
metadata_path = Path('data/metadata/kinase_labels.csv')
df = pd.read_csv(metadata_path)

print(f'Dataset cargado: {len(df)} estructuras')
print(f'Columnas: {df.columns.tolist()}')
df.head()

## 1. Estadísticas Generales

In [ ]:
print('\n=== ESTADÍSTICAS GENERALES ===')
print(f'\nTotal de estructuras: {len(df)}')
print(f'Kinasas únicas: {df["kinase_name"].nunique()}')
print(f'Familias de kinasas: {df["kinase_family"].nunique()}')

# Estados conformacionales
print('\n--- Estados Conformacionales ---')
conformation_dist = df['conformational_state'].value_counts()
for state, count in conformation_dist.items():
    pct = count / len(df) * 100
    print(f'{state.upper()}: {count} ({pct:.1f}%)')

## 2. Distribución de Estados Conformacionales

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Gráfico de barras
conformation_counts = df['conformational_state'].value_counts()
colors = ['#2ecc71', '#e74c3c']
axes[0].bar(conformation_counts.index, conformation_counts.values, color=colors, alpha=0.7, edgecolor='black')
axes[0].set_title('Distribution of Conformational States', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].grid(axis='y', alpha=0.3)

for i, v in enumerate(conformation_counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# Gráfico de pastel
axes[1].pie(conformation_counts.values, labels=conformation_counts.index, 
            autopct='%1.1f%%', colors=colors, startangle=90)
axes[1].set_title('Proportion of States', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Top Kinasas en el Dataset

In [ ]:
top_kinases = df['kinase_name'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(range(len(top_kinases)), top_kinases.values, color='#3498db', edgecolor='black')
ax.set_yticks(range(len(top_kinases)))
ax.set_yticklabels(top_kinases.index)
ax.set_xlabel('Number of Structures')
ax.set_title('Top 15 Kinases by Structure Count', fontsize=12, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

# Añadir valores en barras
for i, (kinase, count) in enumerate(top_kinases.items()):
    ax.text(count + 0.5, i, str(count), va='center', fontweight='bold')

plt.tight_layout()
plt.show()

print('\nDetalles de las top 15 kinasas:')
for kinase in top_kinases.index:
    kinase_df = df[df['kinase_name'] == kinase]
    active = len(kinase_df[kinase_df['conformational_state'] == 'active'])
    inactive = len(kinase_df[kinase_df['conformational_state'] == 'inactive'])
    print(f'{kinase}: {len(kinase_df)} total (activas: {active}, inactivas: {inactive})')

## 4. Análisis de DFG States

In [ ]:
print('\n=== DFG States ===')
dfg_counts = df['dfg_state'].value_counts()
print(dfg_counts)
print(f'\nDistribución: {(dfg_counts / len(df) * 100).round(1).to_dict()}')

# DFG vs Conformational State
fig, ax = plt.subplots(figsize=(10, 6))
dfg_conformation = pd.crosstab(df['dfg_state'], df['conformational_state'])
dfg_conformation.plot(kind='bar', ax=ax, color=['#e74c3c', '#2ecc71'])
ax.set_title('DFG States vs Conformational State', fontsize=12, fontweight='bold')
ax.set_ylabel('Count')
ax.set_xlabel('DFG State')
ax.legend(title='Conformation')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Resolución Cristalográfica

In [ ]:
resolution = df['resolution'].dropna()

print('\n=== RESOLUCIÓN CRISTALOGRÁFICA ===')
print(f'Estructuras con resolución: {len(resolution)}')
print(f'Estructuras sin resolución: {len(df) - len(resolution)}')
print(f'\nMétricas de resolución (Å):')
print(f'  Media: {resolution.mean():.2f}')
print(f'  Mediana: {resolution.median():.2f}')
print(f'  Mín: {resolution.min():.2f}')
print(f'  Máx: {resolution.max():.2f}')
print(f'  Desv. estándar: {resolution.std():.2f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma
axes[0].hist(resolution, bins=40, color='#9b59b6', alpha=0.7, edgecolor='black')
axes[0].axvline(resolution.mean(), color='r', linestyle='--', linewidth=2, label=f'Mean: {resolution.mean():.2f}Å')
axes[0].axvline(resolution.median(), color='g', linestyle='--', linewidth=2, label=f'Median: {resolution.median():.2f}Å')
axes[0].set_xlabel('Resolution (Å)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Crystal Resolution', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Box plot por conformación
resolution_by_conf = [df[df['conformational_state'] == state]['resolution'].dropna() 
                        for state in ['active', 'inactive']]
axes[1].boxplot(resolution_by_conf, labels=['Active', 'Inactive'])
axes[1].set_ylabel('Resolution (Å)')
axes[1].set_title('Resolution by Conformational State', fontsize=12, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Presencia de Ligandos

In [ ]:
ligand_counts = df['ligand_present'].value_counts()

print('\n=== PRESENCIA DE LIGANDOS ===')
print(f'Con ligandos: {ligand_counts.get(1, 0)} ({ligand_counts.get(1, 0)/len(df)*100:.1f}%)')
print(f'Sin ligandos: {ligand_counts.get(0, 0)} ({ligand_counts.get(0, 0)/len(df)*100:.1f}%)')

# Ligandos vs Conformación
ligand_conformation = pd.crosstab(df['ligand_present'], df['conformational_state'])
print('\nLigandos vs Conformación:')
print(ligand_conformation)

fig, ax = plt.subplots(figsize=(10, 6))
ligand_conformation.T.plot(kind='bar', ax=ax, color=['#95a5a6', '#f39c12'])
ax.set_title('Ligand Presence by Conformational State', fontsize=12, fontweight='bold')
ax.set_ylabel('Count')
ax.set_xlabel('Conformational State')
ax.legend(['No ligand', 'With ligand'], title='Ligand')
plt.xticks(rotation=0)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Análisis de Splits (Train/Val/Test)

In [ ]:
splits_dir = Path('data/splits')
splits_info = {}

if splits_dir.exists():
    for split_name in ['train', 'val', 'test']:
        split_path = splits_dir / f'{split_name}.csv'
        if split_path.exists():
            split_df = pd.read_csv(split_path)
            splits_info[split_name] = split_df
    
    print('\n=== SPLITS ANALYSIS ===')
    split_stats = {}
    
    for split_name, split_df in splits_info.items():
        active = len(split_df[split_df['conformational_state'] == 'active'])
        inactive = len(split_df[split_df['conformational_state'] == 'inactive'])
        total = len(split_df)
        pct = total / len(df) * 100
        
        split_stats[split_name] = {'total': total, 'active': active, 'inactive': inactive}
        
        print(f'\n{split_name.upper()}:')
        print(f'  Total: {total} ({pct:.1f}%)')
        print(f'  Activas: {active} ({active/total*100:.1f}%)')
        print(f'  Inactivas: {inactive} ({inactive/total*100:.1f}%)')
    
    # Visualizar splits
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Totales por split
    split_names = list(split_stats.keys())
    split_totals = [split_stats[s]['total'] for s in split_names]
    colors_splits = ['#3498db', '#e67e22', '#e74c3c']
    
    axes[0].bar(split_names, split_totals, color=colors_splits, alpha=0.7, edgecolor='black')
    axes[0].set_title('Data Points per Split', fontsize=12, fontweight='bold')
    axes[0].set_ylabel('Count')
    axes[0].grid(axis='y', alpha=0.3)
    
    for i, v in enumerate(split_totals):
        axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')
    
    # Distribución de clases por split
    active_counts = [split_stats[s]['active'] for s in split_names]
    inactive_counts = [split_stats[s]['inactive'] for s in split_names]
    
    x = np.arange(len(split_names))
    width = 0.35
    
    axes[1].bar(x - width/2, active_counts, width, label='Active', color='#2ecc71', alpha=0.7, edgecolor='black')
    axes[1].bar(x + width/2, inactive_counts, width, label='Inactive', color='#e74c3c', alpha=0.7, edgecolor='black')
    axes[1].set_xlabel('Split')
    axes[1].set_ylabel('Count')
    axes[1].set_title('Class Distribution per Split', fontsize=12, fontweight='bold')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(split_names)
    axes[1].legend()
    axes[1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print('Splits no encontrados')

## 8. Familias de Kinasas

In [ ]:
print('\n=== FAMILIAS DE KINASAS ===')
kinase_families = df['kinase_family'].value_counts()
print(f'Total familias: {len(kinase_families)}')
print('\nTop familias:')
print(kinase_families.head(10))

fig, ax = plt.subplots(figsize=(12, 6))
kinase_families.head(15).plot(kind='barh', ax=ax, color='#16a085', edgecolor='black')
ax.set_xlabel('Number of Structures')
ax.set_title('Top 15 Kinase Families', fontsize=12, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Matriz de Confusión de Estados DFG vs Conformación

In [ ]:
# Crear matriz de confusión
confusion_matrix = pd.crosstab(
    df['dfg_state'], 
    df['conformational_state']
)

print('\nMatriz de DFG State vs Conformational State:')
print(confusion_matrix)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(confusion_matrix, annot=True, fmt='d', cmap='YlOrRd', ax=ax, cbar_kws={'label': 'Count'})
ax.set_title('DFG State vs Conformational State', fontsize=12, fontweight='bold')
ax.set_xlabel('Conformational State')
ax.set_ylabel('DFG State')
plt.tight_layout()
plt.show()

## 10. Resumen Final

In [ ]:
print('\n' + '='*80)
print('RESUMEN DEL DATASET')
print('='*80)
print(f'\nTotal de estructuras: {len(df)}')
print(f'Kinasas únicas: {df["kinase_name"].nunique()}')
print(f'Familias de kinasas: {df["kinase_family"].nunique()}')
print(f'Especies: {df["species"].nunique()}')

active_total = len(df[df['conformational_state'] == 'active'])
inactive_total = len(df[df['conformational_state'] == 'inactive'])

print(f'\nEstados conformacionales:')
print(f'  Activas: {active_total} ({active_total/len(df)*100:.1f}%)')
print(f'  Inactivas: {inactive_total} ({inactive_total/len(df)*100:.1f}%)')

print(f'\nCon ligandos: {len(df[df["ligand_present"] == 1])} ({len(df[df["ligand_present"] == 1])/len(df)*100:.1f}%)')
print(f'Sin ligandos: {len(df[df["ligand_present"] == 0])} ({len(df[df["ligand_present"] == 0])/len(df)*100:.1f}%)')

resolution = df['resolution'].dropna()
print(f'\nResolución media: {resolution.mean():.2f} Å')
print(f'Resolución rango: {resolution.min():.2f} - {resolution.max():.2f} Å')

print('\n' + '='*80)